# H SC 790 · Lab 5 · Link Three Sources Without a Shared Identifier

**Week 6.** Due Thursday, March 4, 11:59 PM · 30 points.

**What earns full credit**
- Runs end to end from a fresh runtime
- Deterministic and probabilistic results compared
- Recall by subgroup, before and after your change
- AI-use note included

**Rules.** Everything here is synthetic. Never load protected health information or confidential employer data into Colab or GitHub. The lab is finished when it runs top to bottom from a fresh runtime (Runtime → Restart session and run all).

The data dictionary for this lab is the README in the lab's folder of the course repository.

Cells marked `# YOUR CODE HERE` and *Your answer here.* are yours to complete. Add cells freely.

In [ ]:
# Setup: run this cell first, every time.
import os, json, urllib.request
import numpy as np
import pandas as pd

# The course data, read straight from the course repository. Never from your own machine.
COURSE_URL = os.environ.get("HSC790_DATA", "https://raw.githubusercontent.com/OWNER/hsc790-spr27/main/data/")

def read_text(rel):
    """Text of one file in the course data folder, for example read_text('SNAPSHOT.txt')."""
    return urllib.request.urlopen(COURSE_URL + rel).read().decode()

print("Data snapshot:", read_text("SNAPSHOT.txt").strip())

Three files describe overlapping groups of residents, with no identifier in common. Link them with Splink (probabilistic linkage on DuckDB), evaluate against the truth key, and find out who links worst.

The truth key is for evaluation only. It never goes into blocking or matching.

In [ ]:
%pip install -q splink==5.0.0
import logging, warnings
logging.disable(logging.WARNING); warnings.filterwarnings("ignore")          # keep Splink's progress messages out of the notebook
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
import splink.comparison_library as cl

ehr = pd.read_csv(COURSE_URL + "lab05/ehr.csv", dtype=str, keep_default_na=False)
lab = pd.read_csv(COURSE_URL + "lab05/lab.csv", dtype=str, keep_default_na=False)
imm = pd.read_csv(COURSE_URL + "lab05/imm.csv", dtype=str, keep_default_na=False)
truth = pd.read_csv(COURSE_URL + "lab05/truth_key.csv", dtype=str)
for name, df in [("ehr", ehr), ("lab", lab), ("imm", imm)]:
    print(name, len(df)); print(df.head(3).to_string(index=False))

## 1. Profile and standardize

Compare field completeness and formats across the three sources. Then clean names, dates, sex, and ZIP code with one shared function, so that a difference between two records means something.

In [ ]:
def clean(df, date_format):
    """One shared cleaning function. Returns rec_id, first_name, last_name, birth_date (YYYY-MM-DD), sex, zip."""
    # YOUR CODE HERE
    pass

ehr_c, lab_c, imm_c = clean(ehr, "%Y-%m-%d"), clean(lab, "%m/%d/%Y"), clean(imm, "%Y%m%d")
pd.DataFrame({n: d.isna().mean().round(3) for n, d in [("ehr", ehr_c), ("lab", lab_c), ("imm", imm_c)]})

## 2. Try deterministic first

Link EHR to laboratory, and EHR to the immunization registry, on exact first name, last name, birth date, and sex. Use the truth key to say how many of those links are right and what share of the true pairs you found. This is your baseline.

In [ ]:
key = truth.set_index("rec_id").person_key
group = truth.drop_duplicates("person_key").set_index("person_key").race_ethnicity

def true_pairs(left, right):
    """Every pair of records, one from each file, that belong to the same person."""
    l = left.assign(k=left.rec_id.map(key)); r = right.assign(k=right.rec_id.map(key))
    return l.merge(r, on="k", suffixes=("_l", "_r"))[["rec_id_l", "rec_id_r", "k"]]

# YOUR CODE HERE

## 3. Specify the probabilistic model, 4. estimate and predict

Choose a comparison for each field and at least two blocking rules. The slide's model is below as a function. Estimate u by random sampling and m by expectation-maximization, then predict.

In [ ]:
def link(left, right, comparisons=None, rules=None):
    settings = SettingsCreator(
        link_type='link_only', unique_id_column_name='rec_id',
        comparisons=comparisons or [
            cl.NameComparison('last_name'),
            cl.NameComparison('first_name'),
            cl.DateOfBirthComparison('birth_date',
                                     input_is_string=True),
            cl.ExactMatch('sex'),
            cl.ExactMatch('zip')],
        blocking_rules_to_generate_predictions=rules or [
            block_on('birth_date'),
            block_on('last_name', 'zip')])
    db = DuckDBAPI()
    linker = Linker([db.register(left), db.register(right)], settings)
    linker.training.estimate_u_using_random_sampling(max_pairs=1e6)
    linker.training.estimate_parameters_using_expectation_maximisation(
        block_on('birth_date'))
    pairs = linker.inference.predict(threshold_match_probability=0.5)
    return pairs.as_pandas_dataframe(), linker

pairs_lab, linker_lab = link(ehr_c, lab_c)
pairs_imm, linker_imm = link(ehr_c, imm_c)
print(len(pairs_lab), "candidate EHR-lab pairs;", len(pairs_imm), "candidate EHR-registry pairs")
pairs_imm[["match_probability", "match_weight", "rec_id_l", "rec_id_r", "last_name_l", "last_name_r", "first_name_l", "first_name_r"]].sample(5, random_state=1)

Inspect the weights. Do they make sense? Which comparison carries the most evidence, and which the least?

In [ ]:
# YOUR CODE HERE

## 5. Choose thresholds and evaluate

Using the truth key, report precision and recall at three thresholds for each linkage. Pick one and defend it for a named use.

In [ ]:
# YOUR CODE HERE

## 6. Stratify, then improve

Report recall by race and ethnicity for each linkage. Find the group that links worst, look at the true pairs the model missed to see why, change the model, and re-score. Report recall and precision before and after.

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

## Interpretation

Cite your own output in each answer.

**1. How much did probabilistic gain over exact?** In linked people, and in errors of each kind.

*Your answer here.*

**2. Which threshold, for which use?** State the use and which error it can least afford.

*Your answer here.*

**3. Who links worst, and why?** Show recall by subgroup, the cause, and the effect of your fix on recall and precision.

*Your answer here.*

**4. Who should approve this model?** Name the role that should sign off on the threshold and what they need to see.

*Your answer here.*

## AI-use note (required)

Say which AI tools you used, for what, and one thing you checked or corrected in their output. "None" is an acceptable answer.

*Your note here.*

## Submit

1. Runtime → Restart session and run all. Fix anything that fails.
2. File → Save a copy in GitHub, into `labs/lab05/` of your course repository.